In [1]:
import os
import cv2
import numpy as np
import mediapipe as mp

# Paths
DATASET_PATH = "newdata"          # Folder with A-Z folders
OUTPUT_PATH = "MP_Data_Images"    # Where .npy files will be saved

os.makedirs(OUTPUT_PATH, exist_ok=True)

# MediaPipe Hands
mp_hands = mp.solutions.hands

# Create label folders
labels = sorted(os.listdir(DATASET_PATH))
for label in labels:
    os.makedirs(os.path.join(OUTPUT_PATH, label), exist_ok=True)

def extract_keypoints_from_image(results):
    """Extract 21 hand landmarks (x, y, z)"""
    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]
        keypoints = []
        for lm in hand.landmark:
            keypoints.extend([lm.x, lm.y, lm.z])
        return np.array(keypoints)
    else:
        return None

# MediaPipe Hands in IMAGE MODE
with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
) as hands:

    for label in labels:
        label_path = os.path.join(DATASET_PATH, label)
        save_path = os.path.join(OUTPUT_PATH, label)

        img_count = 0

        for img_name in os.listdir(label_path):
            img_path = os.path.join(label_path, img_name)

            image = cv2.imread(img_path)
            if image is None:
                continue

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = hands.process(image_rgb)

            keypoints = extract_keypoints_from_image(results)

            if keypoints is not None:
                np.save(
                    os.path.join(save_path, f"{img_count}.npy"),
                    keypoints
                )
                img_count += 1

        print(f"{label}: {img_count} samples saved")


c:\Users\tejes\OneDrive\Desktop\ISL\myenv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


KeyboardInterrupt: 

In [ ]:
import os
import numpy as np

DATA_PATH = "MP_Data_Images"

labels = sorted(os.listdir(DATA_PATH))  # includes SPACE
label_map = {label: idx for idx, label in enumerate(labels)}

X, y = [], []

for label in labels:
    folder = os.path.join(DATA_PATH, label)
    for f in os.listdir(folder):
        if f.endswith(".npy"):
            X.append(np.load(os.path.join(folder, f)))
            y.append(label_map[label])

X = np.array(X)          # (samples, 63)
y = np.array(y)          # (samples,)
print("X:", X.shape, "y:", y.shape)
print("Labels:", labels)


In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, num_classes=len(labels))

# Train+Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y
)

# Train+Val
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Input(shape=(63,)),
    Dense(96, activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(len(labels), activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)


In [ ]:
model.save("alphabet_space_mlp-new.h5")


In [3]:
import numpy as np
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(
    np.argmax(y_test, axis=1),
    np.argmax(y_pred, axis=1),
    target_names=labels
))


NameError: name 'X_test' is not defined

In [1]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("sign_model_landmark_mlp.h5")

# Labels (must be in SAME order as training)
labels = ['A', 'B', 'Bye', 'C', 'D', 'E', 'F', 'G', 'H', 'Hello', 'I', 'ILoveYou', 'J', 'K', 'L', 'M', 'Meet', 'N', 'No', 'O', 'P', 'Please', 'Q', 'R', 'S', 'T', 'Tell', 'Thankyou', 'U', 'V', 'W', 'X', 'Y', 'Yes', 'Z', 'del', 'space']



In [ ]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)
